# Cas technique — IA appliquée à la radiographie thoracique

**Poste :** Data Scientist / ML Engineer — **Durée indicative :** 2 à 3 jours de travail effectif (à rendre sous 10 jours)

Une startup veut utiliser l'IA pour **détecter des anomalies sur des radiographies du thorax** et **pré-remplir un compte rendu (CR)** que le radiologue relit. Vous disposez de radiographies et des comptes rendus en texte libre rédigés par des radiologues. **Il n'y a pas de labels : la vérité terrain doit être extraite des CR.**

> **Ce qui est évalué :** la qualité et la justification de votre démarche, bien plus que le score final. Un modèle modeste mais rigoureusement évalué vaut mieux qu'un score élevé obtenu avec une fuite de données.

### Conventions
- 📦 **Cellule fournie** : à exécuter telle quelle (vous pouvez la modifier si vous justifiez pourquoi).
- 🧑‍💻 **À compléter** : votre code. Les noms de variables indiqués sont des suggestions ; vous êtes libre de votre structure.
- ❓ **Question** : répondez en Markdown, de façon concise et argumentée (5 à 10 lignes suffisent en général).

Vous pouvez utiliser toute librairie et tout modèle pré-entraîné, à condition de **citer vos sources** et de **vérifier qu'il n'y a pas de fuite** (un modèle pré-entraîné sur ce même jeu de données biaiserait votre évaluation).
L'usage d'assistants IA est autorisé ; vous devez être capable d'expliquer chaque choix lors du débrief.

## Partie 0 — Contexte et cadrage

Le client imagine une architecture **hiérarchique à deux niveaux** :

| Niveau | Tâche | Exigence |
|---|---|---|
| **N1** | Normal vs anormal (binaire) | Doit être **plus fiable** que N2. C'est le filtre de triage : un faux négatif renvoie un patient malade avec un CR « normal ». Cible annoncée « ≈ 97 % » (métrique non précisée). |
| **N2** | Présence de 13 pathologies (+ « No Finding ») | **Multi-label probabiliste** et **calibré** (ex. « 95 % pneumonie, 90 % nodule »). Les dépendances entre pathologies doivent être prises en compte. |

```
 Vue face   ─► Encodeur image ─► p(patho | face)   ─┐
 Vue profil ─► Encodeur image ─► p(patho | profil) ─┼─► Fusion ─► P(anormal), P(patho_1..13) ─► ébauche de CR
 Données cliniques (partielles) ────────────────────┘                    └─► zone d'incertitude ─► relecture médecin
```

Contraintes : un examen peut avoir **0, 1 ou plusieurs vues** ; les données cliniques sont **partielles** ; le système doit **accepter nativement l'absence d'information**, sans imputation naïve.

### Données fournies (`donnees/`)

| Fichier | Contenu |
|---|---|
| `examens.csv` | 1 ligne par examen : `subject_id` (patient), `study_id` (examen), date/heure, nombre d'images, fichier du CR, présence d'un label radiologue |
| `images.csv` | 1 ligne par image : identifiants, **métadonnées DICOM** (`ViewPosition` AP / PA / LATERAL / LL, orientation du patient, description de l'acquisition, taille) — **partiellement manquantes** |
| `images/` | radiographies JPG (redimensionnées depuis le DICOM) |
| `comptes_rendus/` | 1 CR en texte libre (anglais) par examen : `INDICATION` / `HISTORY`, `COMPARISON`, `FINDINGS`, `IMPRESSION`… Les `___` remplacent des informations anonymisées |
| `labels_automatiques_chexpert.csv`, `labels_automatiques_negbio.csv` | labels extraits automatiquement des CR par deux outils publics (CheXpert labeler, NegBio) : 1 / 0 / -1 (incertain) / vide |
| `labels_radiologue.csv` | **labels posés par un radiologue** sur un sous-ensemble d'examens (vérité terrain de meilleure qualité) |

> ⚠️ Données de santé pseudonymisées, transmises **uniquement pour ce test** : ne les redistribuez pas, ne les poussez pas sur GitHub, supprimez-les après le débrief.

❓ **Q0.1** — Le client annonce une cible « ≈ 97 % » pour N1. Quelle(s) métrique(s) proposez-vous de lui présenter à la place (ou en plus), et pourquoi ?

❓ **Q0.2** — Listez les principaux risques de ce projet : cliniques, biais, fuite de données. Lesquels allez-vous pouvoir contrôler dans ce notebook ?

*Vos réponses :*

**Q0.1** —

**Q0.2** —

### Configuration
📦 Cellule fournie. Le **mode rapide** travaille sur un sous-échantillon d'examens et une résolution réduite pour tourner sur CPU en quelques minutes. Passez `MODE_RAPIDE = False` pour vos résultats finaux si votre machine le permet, et indiquez dans votre rendu le mode utilisé.

In [ ]:
import os, re, glob, json, random, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
warnings.filterwarnings("ignore")
import logging; logging.getLogger("pgmpy").setLevel(logging.ERROR)

DATA_DIR    = Path(os.environ.get("DATA_DIR", "donnees"))
REPORTS_DIR = DATA_DIR / "comptes_rendus"       # comptes rendus .txt
IMAGES_DIR  = DATA_DIR / "images"               # radiographies .jpg
CACHE_DIR   = Path(os.environ.get("CACHE_DIR", "cache")); CACHE_DIR.mkdir(exist_ok=True)

MODE_RAPIDE = os.environ.get("MODE_RAPIDE", "1") == "1"
SAMPLE_SIZE = None                              # nombre d'examens à garder (None = tous ; le jeu est petit)
IMG_SIZE    = 320 if MODE_RAPIDE else 512       # côté de l'image en entrée de l'encodeur
SEED = 42

def fixer_graines(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
fixer_graines()
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Mode rapide={MODE_RAPIDE} | examens={SAMPLE_SIZE or 'tous'} | IMG_SIZE={IMG_SIZE} | device={DEVICE}")
assert REPORTS_DIR.exists() and IMAGES_DIR.exists(), "Données introuvables : placez le dossier `donnees/` à la racine (voir README.md)"

### Chargement
📦 Cellule fournie : `cr` (1 ligne par examen, avec les sections du CR découpées), `images` (1 ligne par image, avec métadonnées), et les labels de référence `lab_chexpert`, `lab_negbio`, `lab_radiologue`.

In [ ]:
SECTIONS = {  # en-têtes rencontrés -> section normalisée
    "indication": ["INDICATION", "HISTORY", "CLINICAL HISTORY", "REASON FOR EXAM", "REASON FOR EXAMINATION", "CLINICAL INFORMATION"],
    "comparison": ["COMPARISON", "COMPARISONS"],
    "technique":  ["TECHNIQUE", "EXAMINATION", "EXAM", "STUDY", "TYPE OF EXAMINATION", "WET READ", "NOTIFICATION"],
    "impression": ["IMPRESSION", "CONCLUSION"],
}
ENTETE = re.compile(r"^\s*([A-Z][A-Z /]{2,40}):", re.M)

def decouper_cr(texte):
    """Découpe un CR en sections ; le texte hors en-tête connu est rangé dans `findings`."""
    out = {k: "" for k in ["indication", "comparison", "technique", "findings", "impression"]}
    texte = texte.replace("FINAL REPORT", "")
    morceaux = ENTETE.split(texte)                    # [avant, titre1, corps1, titre2, corps2…]
    out["findings"] = morceaux[0]
    for titre, corps in zip(morceaux[1::2], morceaux[2::2]):
        cle = next((k for k, v in SECTIONS.items() if titre.strip() in v), "findings")
        out[cle] += " " + corps
    return {k: re.sub(r"\s+", " ", v).strip() for k, v in out.items()}

examens = pd.read_csv(DATA_DIR / "examens.csv")
images  = pd.read_csv(DATA_DIR / "images.csv")
lab_chexpert   = pd.read_csv(DATA_DIR / "labels_automatiques_chexpert.csv").set_index("study_id")
lab_negbio     = pd.read_csv(DATA_DIR / "labels_automatiques_negbio.csv").set_index("study_id")
lab_radiologue = pd.read_csv(DATA_DIR / "labels_radiologue.csv").set_index("study_id")

lignes = []
for e in examens.itertuples():
    brut = (REPORTS_DIR / e.fichier_compte_rendu).read_text(errors="ignore")
    lignes.append(dict(uid=e.study_id, subject_id=e.subject_id, date=e.StudyDate, heure=e.StudyTime, texte_brut=brut, **decouper_cr(brut)))
cr = pd.DataFrame(lignes)
if SAMPLE_SIZE:
    cr = cr.sample(n=min(SAMPLE_SIZE, len(cr)), random_state=SEED).reset_index(drop=True)

images = images.rename(columns={"study_id": "uid", "dicom_id": "image_id"})
images = images[images.uid.isin(cr.uid)].reset_index(drop=True)
images["path"] = images.fichier_image.map(lambda f: IMAGES_DIR / f)
cr["images"] = cr.uid.map(images.groupby("uid").image_id.apply(list)).apply(lambda v: v if isinstance(v, list) else [])
print(f"{cr.subject_id.nunique()} patients, {len(cr)} examens, {len(images)} images, {lab_radiologue.shape[0]} examens labellisés par un radiologue")
cr[["uid", "subject_id", "indication", "findings", "impression"]].head(3)

## Partie 1 — Exploration des données (EDA)

🧑‍💻 Explorez les données et **signalez tout ce qui doit influencer la suite** (choix de modèle, de split, de labels). Au minimum :
1. **Volumétrie** : examens, images par examen, distribution des vues (l'information de vue est-elle disponible ?).
2. **Qualité des images** : résolutions, ratios d'aspect, inversions, annotations incrustées (marqueurs L/R, textes), doublons.
3. **Comptes rendus** : structure des sections, longueur, négations, formulations incertaines, comparaisons à l'antérieur, anonymisation (`___`).
4. **Données cliniques et métadonnées** : que peut-on extraire de l'indication (sexe, âge, symptômes, antécédents) et des métadonnées DICOM ? Taux de données manquantes.
5. **Patients** : combien d'examens par patient ? Quelles conséquences ?

In [ ]:
# 🧑‍💻 À compléter — volumétrie et images

In [ ]:
# 🧑‍💻 À compléter — comptes rendus et données cliniques

❓ **Q1** — Quels constats de l'EDA vont directement contraindre vos choix dans la suite (labels, split, prétraitement, fusion) ?

*Votre réponse :*

## Partie 2 — Construction de la vérité terrain à partir des CR (NLP)

Référentiel de labels : **CheXpert** (13 pathologies + `No Finding`) :

`Enlarged Cardiomediastinum, Cardiomegaly, Lung Opacity, Lung Lesion, Edema, Consolidation, Pneumonia, Atelectasis, Pneumothorax, Pleural Effusion, Pleural Other, Fracture, Support Devices`

🧑‍💻 À faire :
1. Écrire un **extracteur de labels** (règles, NegEx-like, modèle NLP… au choix) qui produit pour chaque examen et chaque label un état parmi **positif (1) / négatif (0) / incertain (-1) / non mentionné (NaN)**. Il doit gérer la **négation** (« no pneumothorax ») et l'**incertitude** (« cannot exclude pneumonia », « possible effusion »).
2. **Valider l'extracteur** contre les **labels radiologue** (`labels_radiologue.csv`, l'étalon) et comparez-le aux labelers automatiques **CheXpert** et **NegBio** fournis. Attention : `labels_radiologue` utilise `Airspace Opacity` là où CheXpert dit `Lung Opacity`.
3. Définir le **label N1 « anormal »** de façon cohérente avec N2.
4. Choisir une **stratégie pour les labels incertains** (U-Ones, U-Zeros, ignorer dans la loss, label souple…) et la justifier.

In [ ]:
PATHOLOGIES = ["Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity", "Lung Lesion", "Edema",
               "Consolidation", "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion",
               "Pleural Other", "Fracture", "Support Devices"]
LABELS = ["No Finding"] + PATHOLOGIES

In [ ]:
# 🧑‍💻 À compléter — extracteur de labels
def extraire_labels(texte: str) -> dict:
    """Retourne {label: 1 | 0 | -1 | NaN} pour chaque label de LABELS."""
    raise NotImplementedError

# labels = pd.DataFrame([extraire_labels(f + " " + i) for f, i in zip(cr.findings, cr.impression)], index=cr.uid)

In [ ]:
# 🧑‍💻 À compléter — validation de l'extracteur (vs radiologue, vs CheXpert / NegBio) + label N1 « anormal »

❓ **Q2.1** — Quelle stratégie avez-vous retenue pour les labels incertains, et quel effet attendez-vous sur l'entraînement et sur l'évaluation ?

❓ **Q2.2** — Que vous apprend la comparaison avec les labels radiologue et avec CheXpert / NegBio sur la fiabilité de votre vérité terrain ? Quelles pathologies vous semblent trop bruitées pour être évaluées sérieusement ?

*Vos réponses :*

**Q2.1** —

**Q2.2** —

## Partie 3 — Nettoyage et préparation

🧑‍💻 À faire :
1. **Prétraitement des images** : normalisation, redimensionnement, gestion du ratio d'aspect. Le client parle de 512 → ~724 px : discutez du compromis détail / coût (une fonction de base vous est fournie ci-dessous).
2. **Identification de la vue** (face / profil) : `ViewPosition` est **manquant pour une partie des images**. Que faites-vous de ces images ? Proposez une méthode et vérifiez-la.
3. **Split train / validation / test** sans fuite (indice : regardez `subject_id`). Avec si peu de patients, comment rendre l'évaluation fiable ?
4. **Déséquilibre des classes** : quantifiez-le et proposez une stratégie.

📦 L'extraction des embeddings est fournie pour que le temps de calcul ne soit pas un frein : un **encodeur pré-entraîné ImageNet, gelé** (`densenet121`). Libre à vous de le remplacer ou de faire du fine-tuning si vous avez un GPU (bonus, non requis). Justifiez votre choix.

In [ ]:
import timm
from PIL import Image
from torchvision import transforms as T

def charger_image(path, size=IMG_SIZE):
    """Niveaux de gris -> carré `size` en conservant le ratio (padding noir) -> 3 canaux normalisés ImageNet."""
    im = Image.open(path).convert("L")
    w, h = im.size; s = size / max(w, h)
    im = im.resize((max(1, round(w * s)), max(1, round(h * s))), Image.BILINEAR)
    canvas = Image.new("L", (size, size)); canvas.paste(im, ((size - im.size[0]) // 2, (size - im.size[1]) // 2))
    x = T.functional.to_tensor(canvas).repeat(3, 1, 1)
    return T.functional.normalize(x, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

ENCODEUR = "densenet121.tv_in1k"
encodeur = timm.create_model(ENCODEUR, pretrained=True, num_classes=0).eval().to(DEVICE)

@torch.no_grad()
def extraire_embeddings(paths, batch_size=32):
    cache = CACHE_DIR / f"emb_{ENCODEUR}_{IMG_SIZE}.npz"
    deja = dict(np.load(cache, allow_pickle=True)) if cache.exists() else {}
    a_faire = [p for p in paths if Path(p).stem not in deja]
    for i in range(0, len(a_faire), batch_size):
        lot = a_faire[i:i + batch_size]
        z = encodeur(torch.stack([charger_image(p) for p in lot]).to(DEVICE)).cpu().numpy()
        deja.update({Path(p).stem: v for p, v in zip(lot, z)})
        if i % (batch_size * 20) == 0: print(f"  {i + len(lot)}/{len(a_faire)}")
    if a_faire: np.savez(cache, **deja)
    return np.stack([deja[Path(p).stem] for p in paths])

EMB = extraire_embeddings(list(images.path))     # (n_images, 1024), aligné sur `images`
print("Embeddings :", EMB.shape)

In [ ]:
# 🧑‍💻 À compléter — identification de la vue (face / profil) pour chaque image

In [ ]:
# 🧑‍💻 À compléter — split train / val / test sans fuite + analyse du déséquilibre

❓ **Q3.1** — Votre split garantit-il l'absence de fuite ? Que reste-t-il comme risque, et comment l'avez-vous réduit ?

❓ **Q3.2** — Quelle résolution d'entrée recommanderiez-vous au client (512, 724, 768…) et pourquoi ?

*Vos réponses :*

**Q3.1** —

**Q3.2** —

## Partie 4 — Modèle niveau 1 (normal / anormal)

🧑‍💻 À faire :
1. Un modèle N1 sur les embeddings (ou fine-tuné). Pour cette partie, une fusion simple des vues suffit (ex. moyenne des embeddings disponibles) ; la Partie 6 approfondit.
2. **Choix du seuil** sur la validation, selon une **sensibilité cible** que vous justifiez.
3. Sur le test : AUC, sensibilité, spécificité, VPN, matrice de confusion, **avec intervalles de confiance**.
4. Comparez avec un N1 **dérivé de N2** (`P(anormal) = 1 − Π(1 − p_i)` ou max) une fois la Partie 5 faite.

In [ ]:
# 🧑‍💻 À compléter — modèle N1, choix du seuil, évaluation

❓ **Q4** — Comment avez-vous choisi votre sensibilité cible et votre seuil ? Que répondez-vous au client qui demande « 97 % » ?

*Votre réponse :*

## Partie 5 — Modèle niveau 2 (multi-label, 13 pathologies)

🧑‍💻 À faire :
1. Une tête multi-label (sigmoïde) avec une loss adaptée au déséquilibre et à votre stratégie pour les labels incertains.
2. AUC par pathologie (avec effectifs et, idéalement, IC), F1 macro / micro à des seuils choisis sur la validation.
3. **Calibration** : reliability diagram, ECE, puis correction (ex. temperature scaling) apprise sur la validation.
4. Retour à la Partie 4 : N1 dérivé de N2 vs N1 dédié. Lequel est le plus fiable, et pourquoi ?

In [ ]:
# 🧑‍💻 À compléter — modèle N2 multi-label

In [ ]:
# 🧑‍💻 À compléter — calibration (reliability diagram, ECE, temperature scaling) + comparaison N1 dédié vs dérivé

❓ **Q5** — Vos probabilités N2 sont-elles exploitables telles quelles par un médecin (« 95 % pneumonie ») ? Quelles pathologies ne devriez-vous **pas** afficher au client en l'état ?

*Votre réponse :*

## Partie 6 — Multi-vues et données manquantes

🧑‍💻 À faire :
1. Une stratégie de fusion qui accepte un **nombre variable de vues** (0, 1, 2+), sans imputer de fausse image.
2. L'intégration des **données cliniques partielles** issues de `INDICATION`, sans imputation naïve.
3. **Expérience obligatoire** : sur le test, **retirez artificiellement** une vue (face, puis profil) et les données cliniques, et mesurez la dégradation (AUC N1 et AUC moyenne N2). Comparez au moins deux stratégies de fusion.

In [ ]:
# 🧑‍💻 À compléter — fusion multi-vues + données cliniques

In [ ]:
# 🧑‍💻 À compléter — expérience de retrait (vue face, vue profil, clinique)

❓ **Q6** — Pourquoi votre méthode de fusion « accepte-t-elle nativement » l'absence d'une vue ou d'une donnée clinique ? En quoi est-ce différent d'une imputation ?

*Votre réponse :*

## Partie 7 — Fusion bayésienne (CNN + réseau bayésien + clinique)

Les pathologies ne sont pas indépendantes (ex. cardiomégalie ↔ épanchement ↔ œdème ; opacité ↔ atélectasie ↔ pneumonie). Le client imagine un **réseau bayésien** en couche de fusion :

- nœud **`Anormal`** parent des pathologies (cohérence N1 / N2) ;
- **dépendances entre pathologies** apprises sur les données et/ou imposées par l'expertise ;
- sorties du modèle image par vue utilisées comme **évidence** (nœuds observés, ou *virtual evidence*) ;
- **données cliniques** et **vues manquantes** = nœuds **non observés**, donc **marginalisés**.

🧑‍💻 À faire :
1. **Mesurez les dépendances** entre labels (co-occurrence, information mutuelle…).
2. **Proposez une structure** (apprentissage de structure type Hill-Climbing / BIC, graphe expert, ou les deux) et justifiez-la.
3. **Branchez les sorties du modèle image** et les données cliniques ; faites l'inférence avec des nœuds manquants.
4. **Comparez** à la baseline sans réseau bayésien : AUC, calibration, robustesse au retrait d'une vue.

Librairie suggérée : `pgmpy`. Vous pouvez restreindre le réseau à un sous-ensemble de pathologies si vous le justifiez (taille, effectifs).
Si vous pensez qu'un réseau bayésien n'est **pas** le bon choix, argumentez et implémentez une alternative (classifier chains, têtes conditionnelles, CRF…).

In [ ]:
# 🧑‍💻 À compléter — dépendances entre pathologies (co-occurrence, information mutuelle)

In [ ]:
# 🧑‍💻 À compléter — structure du réseau, branchement des sorties image et de la clinique, inférence, comparaison

❓ **Q7.1** — Quelle structure avez-vous retenue et comment la défendriez-vous devant un radiologue ?

❓ **Q7.2** — Le réseau bayésien apporte-t-il réellement quelque chose par rapport à la fusion neuronale ? Sur quels critères (AUC, calibration, robustesse, interprétabilité) ?

*Vos réponses :*

**Q7.1** —

**Q7.2** —

## Partie 8 — Restitution et interprétabilité

🧑‍💻 À faire :
1. Une **sortie par examen** au format du client : `P(anormal)`, top-k pathologies avec probabilités, drapeau **« à relire »** pour la zone d'incertitude (définissez-la).
2. **Cartes d'activation** (Grad-CAM ou équivalent) sur quelques cas, dont au moins un **échec**. Que regarde le modèle ?
3. Une **ébauche de CR** (« normal » ou liste des pathologies probables), clairement présentée comme une **aide à la relecture**.

In [ ]:
# 🧑‍💻 À compléter — sortie par examen, zone « à relire », ébauche de CR

In [ ]:
# 🧑‍💻 À compléter — Grad-CAM sur quelques cas (succès et échecs)

❓ **Q8** — Que montrent vos cartes d'activation ? Feriez-vous confiance à ce modèle sur un cas où la carte est hors du poumon ?

*Votre réponse :*

## Partie 9 — Analyse critique

❓ **Q9.1** — Quelles sont les principales limites de votre travail (labels, données, protocole, modèle) ? Classez-les par impact.

❓ **Q9.2** — Que faudrait-il **avant un usage clinique** de ce système ? (validation, réglementaire, organisation, suivi en production)

❓ **Q9.3** — Si vous aviez deux semaines de plus et un GPU, que feriez-vous en priorité, et pourquoi ?

*Vos réponses :*

**Q9.1** —

**Q9.2** —

**Q9.3** —

---
### Rendu
Avant de rendre : `Kernel → Restart & Run All` doit tourner sans erreur dans le mode indiqué. Poussez le notebook **avec ses sorties**, puis suivez les instructions de rendu du `README.md`.